# 01 - Cierre heuristico seguro

Este notebook deja visible la baseline de cierre de oracion de `realtime/`: que decide cuando commitear texto, cuando esperar y cuando marcar baja confianza.

La idea no es demostrar que la heuristica sea el modelo final, sino dejar una base segura y medible para compararla despues contra un provider LLM real.

## 1. Setup reproducible

Se importan solamente modulos locales y librerias de notebook. No se usa GPU, VM, Ollama ni API externa.

In [1]:
from pathlib import Path
import csv
import json
import sys
from collections import Counter
from IPython.display import Markdown, display

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "realtime" / "src").exists():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def _fmt(value):
    if isinstance(value, float):
        return f"{value:.4f}"
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value) or "-"
    text = str(value)
    return text.replace("\n", " ").replace("|", "\\|")


def show_table(rows, columns):
    if not rows:
        display(Markdown("_Sin filas para mostrar._"))
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(_fmt(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))


def show_json(payload):
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

print(f"Repo detectado: {ROOT.name}")
from realtime.src.contracts import PartialHypothesis, CommitAction, CommitDecision, CorrectionResult
from realtime.src.cierre import HeuristicClosureProvider
from realtime.src.corrector import IdentityCorrectionProvider
from realtime.src.evaluar_cierre import DEMO_CASES, evaluate_closure
from realtime.src.validation import validate_commit_decision, validate_correction_result

closure = HeuristicClosureProvider()
corrector = IdentityCorrectionProvider()
print("Provider de cierre:", closure.name)
print("Provider de correccion:", corrector.name)

Repo detectado: labios-argentos
Provider de cierre: heuristic_closure
Provider de correccion: identity_correction


## 2. Contrato que debe cumplir el cierre

El loop principal espera siempre una decision estructurada. La regla de seguridad mas importante es: si la accion no es `commit`, `committed_text` queda vacio.

In [2]:
sample = PartialHypothesis(
    partial_text="ayer fuimos a la cancha y estuvo buenisimo.",
    segment_id="demo_contract",
    source="notebook",
)
decision = closure.decide(sample)
validated, fallback_used = validate_commit_decision(decision)

show_table(
    [
        {
            "texto_entrada": sample.partial_text,
            "action": validated.action.value,
            "committed_text": validated.committed_text,
            "confidence": validated.confidence,
            "reason": validated.reason,
            "risk_flags": validated.risk_flags,
            "fallback": fallback_used,
        }
    ],
    ["texto_entrada", "action", "committed_text", "confidence", "reason", "risk_flags", "fallback"],
)


| texto_entrada | action | committed_text | confidence | reason | risk_flags | fallback |
| --- | --- | --- | --- | --- | --- | --- |
| ayer fuimos a la cancha y estuvo buenisimo. | commit | ayer fuimos a la cancha y estuvo buenisimo. | 0.8600 | puntuacion_de_cierre | - | False |

## 3. Casos borde principales

Estos casos cubren el comportamiento que queremos defender: texto vacio o roto no se commitea, conectores colgantes esperan, repeticiones fuertes van a baja confianza y solo se commitea cuando hay una senal razonable de cierre.

In [3]:
rows = []
for idx, case in enumerate(DEMO_CASES, start=1):
    hypothesis = PartialHypothesis(
        partial_text=case["partial_text"],
        segment_id=f"demo_{idx:02d}",
        source="notebook_demo",
    )
    decision = closure.decide(hypothesis)
    rows.append({
        "caso": case["case"],
        "texto parcial": case["partial_text"] or "<vacio>",
        "esperado": case["expected_action"],
        "predicho": decision.action.value,
        "ok": case["expected_action"] == decision.action.value,
        "confianza": decision.confidence,
        "razon": decision.reason,
        "riesgos": decision.risk_flags,
    })

show_table(rows, ["caso", "texto parcial", "esperado", "predicho", "ok", "confianza", "razon", "riesgos"])

| caso | texto parcial | esperado | predicho | ok | confianza | razon | riesgos |
| --- | --- | --- | --- | --- | --- | --- | --- |
| empty | <vacio> | low_confidence | low_confidence | True | 0.0000 | texto_vacio | empty |
| dangling | yo creo que | wait | wait | True | 0.7200 | conector_colgante | dangling_connector |
| dangling | me parece que | wait | wait | True | 0.7200 | conector_colgante | dangling_connector |
| repetition | bueno bueno bueno bueno | low_confidence | low_confidence | True | 0.2000 | texto_repetitivo | repetition |
| punctuated | ayer fuimos a la cancha y estuvo buenisimo. | commit | commit | True | 0.8600 | puntuacion_de_cierre | - |
| long_enough | vos tenes razon che gracias por avisarme hoy | commit | commit | True | 0.6600 | contexto_suficiente_sin_conector_colgante | heuristic_commit |
| incomplete | no se si | wait | wait | True | 0.7200 | conector_colgante | dangling_connector |

## 4. Metricas de la baseline

La evaluacion sintetica es chica a proposito: sirve como smoke test explicable antes de comparar contra salidas reales del VSR.

In [4]:
summary = evaluate_closure(DEMO_CASES, closure)
metric_rows = [
    {"metrica": "casos", "valor": summary["count"]},
    {"metrica": "accuracy", "valor": summary["accuracy"]},
    {"metrica": "precision_commit", "valor": summary["commit_precision"]},
    {"metrica": "recall_commit", "valor": summary["commit_recall"]},
    {"metrica": "commits_prematuros", "valor": summary["premature_commit_rate"]},
    {"metrica": "waits_innecesarios", "valor": summary["unnecessary_wait_rate"]},
    {"metrica": "recall_low_confidence", "valor": summary["low_confidence_recall"]},
    {"metrica": "latencia_p50_ms", "valor": summary["latency_ms"]["p50"]},
    {"metrica": "latencia_p95_ms", "valor": summary["latency_ms"]["p95"]},
    {"metrica": "fallbacks", "valor": summary["fallbacks"]},
]
show_table(metric_rows, ["metrica", "valor"])

| metrica | valor |
| --- | --- |
| casos | 7 |
| accuracy | 1.0000 |
| precision_commit | 1.0000 |
| recall_commit | 1.0000 |
| commits_prematuros | 0.0000 |
| waits_innecesarios | 0.0000 |
| recall_low_confidence | 1.0000 |
| latencia_p50_ms | 0.0748 |
| latencia_p95_ms | 0.1225 |
| fallbacks | 0 |

## 5. Fallbacks defensivos

Si un provider futuro devuelve algo invalido, el sistema debe caer a `wait`. Para correccion, el fallback conserva el texto crudo.

In [5]:
invalid_commit = CommitDecision(action=CommitAction.COMMIT, committed_text="", confidence=0.9, reason="mal formado")
validated_commit, commit_fallback = validate_commit_decision(invalid_commit)

invalid_correction = CorrectionResult(raw_text="texto original", corrected_text="", confidence=0.9, changed=True)
validated_correction, correction_fallback = validate_correction_result(invalid_correction, "texto original")

show_table(
    [
        {
            "caso": "commit invalido sin texto",
            "fallback_usado": commit_fallback,
            "salida_segura": validated_commit.action.value,
            "texto_final": validated_commit.committed_text or "<vacio>",
            "reason": validated_commit.reason,
            "risk_flags": validated_commit.risk_flags,
        },
        {
            "caso": "correccion invalida vacia",
            "fallback_usado": correction_fallback,
            "salida_segura": "texto crudo",
            "texto_final": validated_correction.corrected_text,
            "reason": "empty_corrected_text",
            "risk_flags": validated_correction.risk_flags,
        },
    ],
    ["caso", "fallback_usado", "salida_segura", "texto_final", "reason", "risk_flags"],
)


| caso | fallback_usado | salida_segura | texto_final | reason | risk_flags |
| --- | --- | --- | --- | --- | --- |
| commit invalido sin texto | True | wait | <vacio> | invalid_commit_without_text | fallback |
| correccion invalida vacia | True | texto crudo | texto original | empty_corrected_text | fallback, empty_corrected_text |

## 6. Lectura final

Resultado de este notebook:

- La baseline corre localmente sin dependencias pesadas.
- Las decisiones salen con contrato estricto y validacion.
- Ante duda o salida invalida, el sistema espera en vez de inventar.
- Esta baseline ya es comparable contra un provider LLM/Ollama futuro.